# local — SQL Warehouse execution

In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
CONFIG = ROOT / 'examples/e2e/config.json'
PAIR = 'orders'
RUN_ID = os.getenv('DTRACK_RUN_ID', 'nb-local')
EXECUTE = False  # Needs DTRACK_DBX_PROFILE and DTRACK_DBX_WAREHOUSE_ID.

subprocess.run([sys.executable, str(ROOT / 'examples/e2e/make_synthetic.py')], cwd=ROOT, check=True, capture_output=True)
pair = json.loads(CONFIG.read_text())['pairs'][PAIR]
{'left_file': pair['left']['path'], 'right_table': '.'.join(pair['right'][k] for k in ('catalog', 'schema', 'table'))}

In [ ]:
def dtrack(*args, execute=EXECUTE):
    command = [sys.executable, '-m', 'dtrack', *map(str, args)]
    print('$', shlex.join(command))
    return subprocess.run(command, cwd=ROOT, text=True, check=execute) if execute else command

# Local mode has no remote watcher Job: Left is local; Right is queried through SQL Warehouse.
preflight = dtrack('local', 'preflight', '--config', CONFIG, '--pair', PAIR, '--run-id', RUN_ID)
preflight

In [ ]:
# This checks that Left and Right select the same deterministic keys before a full local run.
key_check = dtrack(
    'local', 'key-check', '--config', CONFIG, '--pair', PAIR,
    '--from-date', pair['fromDate'], '--to-date', pair['toDate']
)
key_check

In [ ]:
run = dtrack('local', 'run', '--config', CONFIG, '--pair', PAIR, '--run-id', RUN_ID, '--gate', 'auto')
artifacts = {
    'report': ROOT / '.dtrack-local/runs' / RUN_ID / 'report',
    'landing': ROOT / '.dtrack-local/runs' / RUN_ID / '_local/data.duckdb',
    'errors': ROOT / '.dtrack-local/runs' / RUN_ID / '_local/logs/errors.log',
    'audit': ROOT / '.dtrack-local/runs' / RUN_ID / 'audit.txt',
}
{'command': run, 'artifacts': {k: str(v) for k, v in artifacts.items()}}

In [ ]:
# Offline unit contract: the complete shared protocol runs on the same real CSV case without Databricks credentials.
import tempfile
sys.path.insert(0, str(ROOT / 'tests'))
from support import pairs_config
from dtrack_left.checks.col_check import ColCheckPipeline, LeftRuntime, publish_fixed_keys, wait_for_manifests
from dtrack_left.checks.row_check import run_row_check
from dtrack_local.csv_target import CsvAnchorTarget
from dtrack_right.backends.store import VolumeStore
from dtrack_right.checks.col_check import ColCheck
from dtrack_right.checks.row_check import run_row_check as run_right_row_check
from dtrack_right.report import build_report

with tempfile.TemporaryDirectory() as directory:
    temp = Path(directory)
    data = ROOT / 'examples/e2e/data'
    store = VolumeStore(temp / 'runs/real')
    test_config = pairs_config(data / 'left')
    target = lambda config: CsvAnchorTarget(config, data / 'right', temp / 'right-work')
    run_id, _ = run_row_check(test_config, store, run_id='real')
    assert run_right_row_check(store, target) == 2
    assert store.get_json('row_check/orders.json')['summary'] == {'match': 56, 'count_diff': 1, 'left_only': 1, 'right_only': 1}
    manifests = wait_for_manifests(store, timeout=1, poll_interval=0)
    for item in manifests: publish_fixed_keys(store, item, temp / 'left-work')
    runtime = LeftRuntime(temp / 'left-work', store)
    assert ColCheckPipeline(store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats).run(manifests) == 11
    results = ColCheck(store, target, poll_interval=0).run()
    verdicts = {(item['pair'], item['vintage'], item['column']): item['verdict'] for item in results}
    assert len(verdicts) == 11
    assert verdicts[('orders', '2026-01-01', 'STATUS')] == 'diff'
    assert verdicts[('orders', '2026-02-01', 'AMOUNT')] == 'diff'
    files = build_report(store, temp / 'report')
    for name, path in files.items(): store.put_bytes(f'report/{name}', path.read_bytes())
    store.put_json('_STATUS.json', {'state': 'complete'})
    assert store.get_bytes('report/report.xlsx') == files['report.xlsx'].read_bytes()
{'run_id': run_id, 'units': len(results), 'diffs': sorted(key for key, verdict in verdicts.items() if verdict == 'diff')}